# Module 00 — Prerequisites & Python Refresher

**Prerequisites:** Basic Python (variables, loops, functions, classes)
**Time:** ~30 minutes
**Part of:** PyTorch From Zero to Practitioner

## Learning Objectives

By the end of this notebook you will be able to:

- Recognize the small set of Python features PyTorch code leans on heavily: classes and `__init__`/`super()`, `*args`/`**kwargs`, list comprehensions, context managers (`with`), and iterators.
- Explain why PyTorch is built around **classes** rather than plain functions.
- Read a "for batch in loader:" style loop and know exactly what it's iterating over.
- Understand, at a high level, what "array-based numerical computing" (NumPy-style thinking) means, since PyTorch tensors extend this idea.

## Why This Matters

You don't need to be a Python expert to learn PyTorch, but PyTorch's API assumes fluency with a handful of specific patterns. If those patterns are shaky, every PyTorch notebook you read afterward will feel harder than it needs to be. This module is a fast, targeted refresher — not a full Python course — aimed exactly at what you'll see starting in the very next notebook.

If everything below already looks familiar, skim it in five minutes and move on. If any section feels new, slow down here — it will pay off immediately.


## 1. Classes, `__init__`, and `super()`

Almost every model you build in PyTorch is a **class** that inherits from `torch.nn.Module`. So before touching PyTorch, let's make sure the underlying Python mechanics are second nature.


In [1]:
class Animal:
    def __init__(self, name, sound):
        # __init__ runs automatically when you create an object: Animal("Dog", "Woof")
        self.name = name
        self.sound = sound

    def speak(self):
        return f"{self.name} says {self.sound}"


class Dog(Animal):
    def __init__(self, name):
        # super().__init__(...) calls the PARENT class's __init__.
        # This lets Dog reuse Animal's setup logic instead of duplicating it.
        super().__init__(name, sound="Woof")
        self.legs = 4


rex = Dog("Rex")
print(rex.speak())
print(rex.legs)


Rex says Woof
4


**What's happening, and why it matters for PyTorch:**

- `Dog` *inherits* from `Animal`, meaning it gets all of `Animal`'s behavior for free, then adds its own (`self.legs`).
- `super().__init__(...)` is how a child class asks its parent to do its own setup first.

This is *exactly* the pattern you will type dozens of times in PyTorch:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()   # let nn.Module do its internal setup
        self.layer = nn.Linear(10, 1)   # then add your own stuff
```

If `super().__init__()` looked unfamiliar above, re-read this section before continuing — it is the single most important piece of "plain Python" background for PyTorch.


### 🔮 Predict before you run

What do you think happens if we forget `super().__init__(name, sound="Woof")` inside `Dog.__init__`, and instead just write `self.legs = 4`? Will `rex.speak()` still work?

Think about it, then check the cell below.


In [2]:
class BrokenDog(Animal):
    def __init__(self, name):
        # No super().__init__() call!
        self.legs = 4

broken = BrokenDog("Fido")
try:
    print(broken.speak())
except AttributeError as e:
    print("Error:", e)


Error: 'BrokenDog' object has no attribute 'name'


**Explanation:** `speak()` is inherited from `Animal`, but it depends on `self.name` and `self.sound`, which are only set inside `Animal.__init__`. Skipping `super().__init__()` means that setup never runs, so those attributes don't exist.

This is precisely why forgetting `super().__init__()` in a PyTorch model produces confusing errors — `nn.Module` sets up internal bookkeeping (like the dictionary that tracks your layers) in its own `__init__`, and skipping it silently breaks things later.


## 2. Iterating and unpacking

PyTorch training loops are built almost entirely out of `for` loops that unpack tuples. Make sure this feels natural:


In [2]:
pairs = [(1, "a"), (2, "b"), (3, "c")]

for number, letter in pairs:
    print(number, letter)


1 a
2 b
3 c


This exact shape — `for x, y in something:` — is what a PyTorch training loop looks like:

```python
for inputs, targets in train_loader:
    ...
```

`train_loader` yields tuples of `(inputs, targets)` one batch at a time, and the loop unpacks each tuple into two variables, just like `pairs` above.


## 3. `with` blocks (context managers)

You'll frequently see code like:

```python
with torch.no_grad():
    ...
```

A `with` block guarantees that some setup happens before the indented code runs, and some cleanup happens after — even if an error occurs inside. A simple built-in example is file handling:


In [4]:
with open("/tmp/demo.txt", "w") as f:
    f.write("hello")
# the file is automatically closed here, even if writing had failed

print("File closed:", f.closed)


File closed: True


`torch.no_grad()` works the same way: it turns *off* gradient tracking for everything inside the block, then automatically turns it back on when the block ends. You'll see exactly why that matters in the autograd notebook.


## 4. NumPy-style thinking (a preview)

If you've used NumPy, PyTorch tensors will feel immediately familiar — same idea, different name. If you haven't, here's the one concept to internalize now:

> Instead of storing numbers in nested Python lists and looping over them one at a time, you store them in a single **array-like object** and operate on the *whole thing at once*.


In [5]:
# The "slow, no PyTorch/NumPy" way
python_list = [1, 2, 3, 4, 5]
doubled = [x * 2 for x in python_list]   # loop over every element
print(doubled)

# The array-based way (NumPy syntax shown here; PyTorch tensors work identically)
import numpy as np
array = np.array([1, 2, 3, 4, 5])
doubled_array = array * 2   # no explicit loop: every element is doubled at once
print(doubled_array)


[2, 4, 6, 8, 10]
[ 2  4  6  8 10]


Why does this matter? Two reasons that will come up constantly:

1. **Speed** — operating on the whole array at once lets the underlying C/C++ code process everything in a tight, optimized loop instead of slow Python bytecode.
2. **Mental model** — once you stop thinking "loop over each number" and start thinking "operate on the whole array/tensor," reading PyTorch code becomes much easier. A line like `predictions = model(batch_of_images)` computes predictions for an entire batch simultaneously — there's no visible loop, but conceptually every image is processed in parallel.

The next notebook picks up exactly here and introduces `torch.Tensor`, the object at the center of everything in PyTorch.


## Exercises

🟢 **Beginner:** Write a class `Vehicle` with `__init__(self, wheels)` and a method `describe()` that returns a string like `"This vehicle has 4 wheels."`. Then write a subclass `Car(Vehicle)` that calls `super().__init__(4)`.

🟡 **Intermediate:** Given `data = [(1, 10), (2, 20), (3, 30)]`, write a `for` loop that unpacks each tuple into `idx, value` and prints `f"item {idx}: {value}"`.

🔴 **Challenge:** Using `with open(...)`, write 3 lines of text to a file, then read them back and print each line stripped of its trailing newline. (This isn't PyTorch-specific — it's here to cement your comfort with `with` blocks before we rely on `torch.no_grad()` later.)


In [6]:
# Space for your exercise solutions



## Common Mistakes

- **Forgetting `super().__init__()`** in a subclass — leads to missing attributes and confusing `AttributeError`s later. In PyTorch specifically, forgetting this in an `nn.Module` subclass causes errors about parameters not being registered.
- **Confusing `self` with the class name** — `self` refers to *this particular instance*, not the class itself. Every method that needs access to an object's own data takes `self` as its first parameter.
- **Assuming array operations loop invisibly, one element at a time in Python** — they don't. That's the whole point: the looping happens in fast, compiled code, not in Python.

## Mental Model

Think of a class as a *blueprint* and `__init__` as the *assembly instructions* that run every time you build a new object from that blueprint. `super().__init__()` means "before doing my own assembly steps, run the parent blueprint's assembly steps first."

## Key Takeaways

- PyTorch models are Python classes that inherit from `nn.Module`, always calling `super().__init__()` first.
- Training loops are `for` loops that unpack `(input, target)` tuples each iteration.
- `with` blocks guarantee setup/cleanup around a block of code — `torch.no_grad()` uses this pattern.
- PyTorch tensors extend the "operate on the whole array at once" idea from NumPy.

## What's Next

**Module 01 — What Is PyTorch, and Your First Tensors** introduces `torch.Tensor` itself: what it is, how it differs from a Python list or a NumPy array, and why it's the foundation everything else in this course is built on.

## Checklist

- [ ] I understand why `super().__init__()` is called inside a subclass's `__init__`
- [ ] I can read a `for x, y in something:` loop and know what's being unpacked
- [ ] I understand what a `with` block guarantees
- [ ] I understand the idea of operating on a whole array at once instead of looping in Python
